# How MedGemma 1.5 sees a CT volume

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rekalantar/MedGemma-CT-Reader/blob/main/tutorial/medgemma_ct.ipynb)

MedGemma 1.5 is the first open multimodal LLM that interprets **volumetric** CT and MRI.
Its vision encoder still only accepts 2D RGB images — so almost everything that makes
volumetric reading work happens in **preprocessing**, before the model sees anything.

That preprocessing is what sections 1–5 are about. They need **no GPU, no account and no
model download** — just NumPy. Section 6 runs the actual model, which does need both.

---

**[The article](ARTICLE_URL) explains the reasoning.** This notebook is the code.

## 1 · Setup

Nothing here needs a GPU.

In [ ]:
%%capture
!pip install -q SimpleITK huggingface_hub

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("ready — sections 1-5 run on CPU")

## 2 · A CT volume, no credentials required

MedGemma's own CT notebook pulls DICOM through Google Cloud's healthcare API, which needs
a Google login on top of a Hugging Face one. We use a NIfTI case from the MedSAM2 demo
set instead: a few MB, public, no auth.

In [ ]:
import SimpleITK as sitk
from huggingface_hub import hf_hub_download

DATASET = "wanglab/CT_DeepLesion-MedSAM2"
CASE = "000009_03_01_036-048"

path = hf_hub_download(DATASET, f"images/{CASE}_0000.nii.gz", repo_type="dataset")
volume_hu = sitk.GetArrayFromImage(sitk.ReadImage(path))      # (z, y, x), Hounsfield

print(f"volume {volume_hu.shape}   HU {volume_hu.min():.0f} to {volume_hu.max():.0f}")

## 3 · Three windows, three channels

A CT stores thousands of distinct Hounsfield values. The encoder takes 256 levels per
channel. Standard practice picks one window, clips to it, and copies the result across
R, G and B — so two channels carry nothing new.

MedGemma gives each channel its own window:

| Channel | HU window | Carries |
|---|---|---|
| Red | −1024 to 1024 | morphological boundaries, bone, lung |
| Green | −135 to 215 | soft tissue |
| Blue | 0 to 80 | subtle low-contrast detail |

These are the values from MedGemma's own CT notebook.

In [ ]:
WINDOWS = ((-1024, 1024), (-135, 215), (0, 80))


def normalise(volume, low, high):
    """Clip to a Hounsfield window and rescale to 0-255."""
    out = np.clip(volume, low, high).astype(np.float32)
    out -= low
    out /= (high - low)
    return out * 255.0


def window_rgb(volume_hu, windows=WINDOWS):
    """(D, H, W) Hounsfield -> (D, H, W, 3) uint8, one window per channel."""
    stacked = np.stack([normalise(volume_hu, lo, hi) for lo, hi in windows], axis=-1)
    return np.round(stacked, 0).astype(np.uint8)


def window_grey(volume_hu, width=400, level=40):
    """The usual single-window greyscale, for comparison."""
    return np.round(normalise(volume_hu, level - width/2, level + width/2), 0).astype(np.uint8)


volume = window_rgb(volume_hu)
print(f"{volume_hu.shape} Hounsfield  ->  {volume.shape} uint8 RGB")

### What each window actually does

Plotting output level against Hounsfield input shows the trade directly: a narrow window
is a steep line over a short range, a wide one is shallow but never saturates.

In [ ]:
hu = np.linspace(-1200, 1300, 1200)

plt.figure(figsize=(9, 4))
for (lo, hi), name, c in zip(WINDOWS,
                             ["red: wide (-1024, 1024)", "green: soft tissue (-135, 215)",
                              "blue: low contrast (0, 80)"],
                             ["#D94F36", "#2E9E5B", "#1B6CA8"]):
    plt.plot(hu, normalise(hu, lo, hi), color=c, linewidth=2, label=name)
plt.plot(hu, normalise(hu, -160, 240), color="#888", linewidth=1.6, linestyle="--",
         label="single greyscale (W400/L40)")

for x, t in [(-1000, "air"), (-90, "fat"), (40, "soft tissue"), (400, "bone")]:
    plt.axvline(x, color="#CCC", linewidth=0.8, zorder=0)
    plt.text(x, 262, t, fontsize=8, ha="center", color="#666")

plt.xlabel("Hounsfield units"); plt.ylabel("output level (0-255)")
plt.ylim(-8, 275); plt.legend(fontsize=9); plt.tight_layout()
plt.savefig("window_response.png", dpi=150, bbox_inches="tight")

In [ ]:
# The same thing as numbers, on tissue values you would actually meet
probe = np.array([-1000, -90, 40, 60, 400, 1000], dtype=np.float32)
labels = ["air", "fat", "soft tissue", "soft tissue +20HU", "bone", "dense bone"]

rgb = window_rgb(probe)
grey = window_grey(probe)
print(f"{'':18}{'HU':>7}{'R':>6}{'G':>6}{'B':>6}{'greyscale':>11}")
for i, name in enumerate(labels):
    r, g, b = rgb[i]
    flag = "  <- saturated" if grey[i] in (0, 255) else ""
    print(f"{name:18}{probe[i]:>7.0f}{r:>6}{g:>6}{b:>6}{grey[i]:>11}{flag}")

sep_blue = abs(int(rgb[3][2]) - int(rgb[2][2]))
sep_grey = abs(int(grey[3]) - int(grey[2]))
print(f"\nTwo soft tissues 20 HU apart:")
print(f"  blue channel     {sep_blue:>3} levels apart")
print(f"  single greyscale {sep_grey:>3} levels apart")

In [ ]:
# And on the real slice
mid = len(volume) // 2
names = ["red: wide", "green: soft tissue", "blue: low contrast", "all three"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for i in range(3):
    axes[i].imshow(volume[mid][..., i], cmap="gray")
    axes[i].set_title(names[i], fontsize=11)
axes[3].imshow(volume[mid])
axes[3].set_title(names[3], fontsize=11)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig("three_windows.png", dpi=150, bbox_inches="tight")

## 4 · Slices become tokens

With windowing settled the volume is just a sequence of RGB images. Each is resized to
896×896 and encoded as **256 vision tokens**.

That is the whole budget problem. MedGemma caps a volume at **85 slices** — 21,760 vision
tokens — which must fit alongside the question and the answer inside a 32K context. A
chest-abdomen CT routinely has three or four hundred slices, so longer volumes are sampled
evenly along z rather than truncated.

In [ ]:
MAX_SLICES, TOKENS_PER_SLICE = 85, 256


def sample_indices(depth, max_slices=MAX_SLICES):
    """Up to `max_slices` indices spread evenly through a volume."""
    if depth <= max_slices:
        return list(range(depth))
    return [round(i / max_slices * (depth - 1)) for i in range(1, max_slices + 1)]


print(f"{'study depth':>12}{'slices sent':>13}{'vision tokens':>15}{'every Nth':>11}")
for depth in (13, 85, 150, 300, 600):
    keep = sample_indices(depth)
    nth = depth / len(keep)
    print(f"{depth:>12}{len(keep):>13}{len(keep)*TOKENS_PER_SLICE:>15,}{nth:>11.1f}")

In [ ]:
# What sampling costs: a small lesion can fall between the slices you keep
depth = 300
keep = set(sample_indices(depth))

plt.figure(figsize=(10, 1.9))
plt.scatter(range(depth), [1]*depth, s=6, color="#DDD", label="slices in the study")
plt.scatter(sorted(keep), [1]*len(keep), s=14, color="#1B8A93", label="slices sent")
for start, size, colour in [(40, 30, "#2E9E5B"), (210, 4, "#D94F36")]:
    hit = sum(1 for i in range(start, start+size) if i in keep)
    plt.plot([start, start+size], [0.9, 0.9], linewidth=5, color=colour,
             label=f"{size}-slice finding: {hit} slice(s) seen")
plt.yticks([]); plt.xlabel("slice index"); plt.ylim(0.8, 1.1)
plt.legend(fontsize=8, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 2.0))
plt.tight_layout()
plt.savefig("sampling_cost.png", dpi=150, bbox_inches="tight")

## 5 · Building the prompt

A volume is not handed over as one input. It becomes an alternating sequence of image
blocks and `SLICE n` labels, bracketed by an instruction and a question. Those labels are
what let the model answer *“hypodense lesion on slices 31–36”* instead of *“somewhere”* —
the difference between an answer you can check and one you must trust.

In [ ]:
import base64, io
import PIL.Image


def encode_slice(slice_rgb, fmt="jpeg"):
    """One uint8 RGB slice -> a base64 data URI."""
    with io.BytesIO() as buf:
        with PIL.Image.fromarray(slice_rgb) as img:
            img.save(buf, format=fmt)
        return f"data:image/{fmt};base64," + base64.b64encode(buf.getbuffer()).decode()


INSTRUCTION = (
    "You are an expert radiologist analysing a contiguous block of axial CT "
    "slices. Review every slice provided below carefully before answering."
)

QUESTION = (
    "Describe the liver in these slices. Note any hypodense lesions, giving the "
    "slice numbers where you see them. If you see none, say so."
)


def build_messages(slices_rgb, question, instruction=INSTRUCTION):
    content = [{"type": "text", "text": instruction}]
    for n, sl in enumerate(slices_rgb, 1):
        content.append({"type": "image", "image": encode_slice(sl)})
        content.append({"type": "text", "text": f"SLICE {n}"})
    content.append({"type": "text", "text": question})
    return [{"role": "user", "content": content}]


messages = build_messages(volume[sample_indices(len(volume))], QUESTION)
blocks = messages[0]["content"]
images = sum(1 for b in blocks if b["type"] == "image")
print(f"{len(blocks)} content blocks: {images} images + {len(blocks)-images} text")
print(f"vision tokens: {images * TOKENS_PER_SLICE:,}")

---

# 6 · Running the model

**Everything above runs on CPU.** This section does not.

You need:

- a **GPU runtime** (Runtime → Change runtime type → GPU)
- a Hugging Face account, the terms accepted on the
  [model page](https://huggingface.co/google/medgemma-1.5-4b-it), and a **read token**
  added as a Colab secret named `HF_TOKEN`

MedGemma 4B is tight on a free T4. The cells below try 16-bit first, fall back to 4-bit
quantisation, and then **measure how many slices actually fit** — which the documentation
does not tell you.

In [ ]:
import os

import torch

HAS_GPU = torch.cuda.is_available()
if not HAS_GPU:
    print("No GPU. Sections 1-5 above are complete on their own; skip the rest,")
    print("or switch to a GPU runtime and rerun from here.")
else:
    GPU = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    MAJOR = torch.cuda.get_device_capability()[0]
    # torch.cuda.is_bf16_supported() returns True on Turing because bf16 is *emulated*.
    # Only Ampere (compute capability 8.0) and later run it natively.
    NATIVE_BF16 = MAJOR >= 8
    print(f"GPU              {GPU}  ({VRAM:.1f} GB)")
    print(f"compute capability {MAJOR}.x")
    print(f"native bfloat16  {NATIVE_BF16}  "
          f"(torch reports {torch.cuda.is_bf16_supported()}, which includes emulation)")

In [ ]:
if HAS_GPU:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("using the HF_TOKEN Colab secret")
    except Exception:
        from huggingface_hub import get_token, notebook_login
        if get_token() is None:
            notebook_login()

In [ ]:
%%capture
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import gc
import time

import transformers

MODEL_ID = "google/medgemma-1.5-4b-it"


def free():
    gc.collect()
    if torch.cuda.is_available():            # these raise on a CPU-only runtime
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def load_model(precision):
    """precision: 'half' (bf16 or fp16) or '4bit'. Returns (processor, model, weights_gb)."""
    free()
    kwargs = {"device_map": "auto"}
    if precision == "4bit":
        kwargs["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["dtype"] = torch.bfloat16 if NATIVE_BF16 else torch.float16

    proc = transformers.AutoProcessor.from_pretrained(MODEL_ID, use_fast=True)
    mdl = transformers.AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs)
    return proc, mdl, torch.cuda.max_memory_allocated() / 1e9


def try_slices(proc, mdl, n, max_new_tokens=64):
    """Run on n slices. Never raises: returns a dict with ok=False instead."""
    free()
    t0 = time.time()
    try:
        idx = sample_indices(len(volume), n)
        inputs = proc.apply_chat_template(
            build_messages(volume[idx], QUESTION), add_generation_prompt=True,
            continue_final_message=False, return_tensors="pt",
            tokenize=True, return_dict=True,
        ).to(mdl.device)
        ntok = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            out = mdl.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
        text = proc.post_process_image_text_to_text(out, skip_special_tokens=True)[0]
        return dict(slices=n, tokens=ntok, ok=True, text=text,
                    peak_gb=torch.cuda.max_memory_allocated() / 1e9,
                    seconds=time.time() - t0)
    except torch.cuda.OutOfMemoryError:
        free()
        return dict(slices=n, ok=False, reason="OOM", seconds=time.time() - t0)
    except Exception as e:
        free()
        return dict(slices=n, ok=False, reason=f"{type(e).__name__}: {str(e)[:70]}",
                    seconds=time.time() - t0)


def sweep(proc, mdl, label, ladder=(1, 2, 4, 6, 8, 12, 16, 24, 32, 48, 64, 85)):
    """Climb the ladder until it runs out of memory. Returns (results, ceiling)."""
    out, ceiling = [], 0
    print(f"{'slices':>7}{'prompt tok':>12}{'peak GB':>10}{'secs':>8}  status")
    for n in ladder:
        r = try_slices(proc, mdl, n)
        out.append(r)
        if r["ok"]:
            ceiling = n
            print(f"{n:>7}{r['tokens']:>12,}{r['peak_gb']:>10.2f}{r['seconds']:>8.1f}  ok")
        else:
            print(f"{n:>7}{'-':>12}{'-':>10}{r['seconds']:>8.1f}  {r['reason']}")
            break
    print(f"\n{label} ceiling: {ceiling} slices of the documented {MAX_SLICES}"
          f"  ({ceiling * TOKENS_PER_SLICE:,} vision tokens)")
    return out, ceiling

### Precision 1 — 16-bit

The default. On a card without native bfloat16 this falls back to float16.

In [ ]:
runs = {}
processor = model = None

if HAS_GPU:
    half = "bf16" if NATIVE_BF16 else "fp16"
    try:
        processor, model, w = load_model("half")
        print(f"{half} weights: {w:.2f} GB of {VRAM:.1f} GB "
              f"({VRAM - w:.2f} GB headroom)\n")
        runs[half] = sweep(processor, model, half)
    except Exception as e:
        print(f"{half} load failed: {type(e).__name__}: {str(e)[:140]}")

### Precision 2 — 4-bit

NF4 quantisation takes the weights from roughly 9 GB to 3 GB. The question is how much of
that headroom turns into extra slices.

In [ ]:
if HAS_GPU:
    try:
        del model, processor
    except NameError:
        pass
    free()
    try:
        processor, model, w = load_model("4bit")
        print(f"4-bit weights: {w:.2f} GB of {VRAM:.1f} GB "
              f"({VRAM - w:.2f} GB headroom)\n")
        runs["4bit"] = sweep(processor, model, "4bit")
    except Exception as e:
        print(f"4-bit load failed: {type(e).__name__}: {str(e)[:140]}")

### The verdict

In [ ]:
print(f"{'precision':>10}{'ceiling':>9}{'of 85':>8}{'vision tokens':>15}")
for label, (res, ceil) in runs.items():
    print(f"{label:>10}{ceil:>9}{ceil / MAX_SLICES:>8.1%}{ceil * TOKENS_PER_SLICE:>15,}")

best_label, (best_res, ceiling) = max(runs.items(), key=lambda kv: kv[1][1]) if runs \
    else ("none", ([], 0))
print(f"\nbest: {best_label} at {ceiling} slices")

plt.figure(figsize=(8, 3.8))
for label, (res, _) in runs.items():
    ok = [r for r in res if r["ok"]]
    if ok:
        plt.plot([r["slices"] for r in ok], [r["peak_gb"] for r in ok],
                 "o-", linewidth=2, label=f"{label}")
if HAS_GPU:
    plt.axhline(VRAM, color="#D94F36", linestyle="--", label=f"{GPU} limit")
plt.xlabel("slices sent"); plt.ylabel("peak GPU memory (GB)")
plt.legend(); plt.tight_layout()
plt.savefig("memory_vs_slices.png", dpi=150, bbox_inches="tight")

### What it said at the best setting

In [ ]:
if ceiling:
    answer = try_slices(processor, model, ceiling, max_new_tokens=300)
    print(f"{ceiling} slices, {answer['tokens']:,} prompt tokens, {answer['seconds']:.0f}s\n")
    print(answer["text"].strip()[-1600:])
else:
    print("The model did not run on this hardware — a result worth reporting.")

### Does the three-window encoding actually help?

This is the question the technical report does not answer. Three windows carry more
*information* than one greyscale repeated three times — but the encoder was pretrained on
photographs, where R, G and B are strongly correlated. Three independent Hounsfield
windows produce colour statistics no photograph ever had.

So it could help, or it could confuse filters tuned to natural colour. Same volume, same
question, two encodings:

In [ ]:
def repeated_grey(volume_hu, width=400, level=40):
    """The common approach: one window, copied across all three channels."""
    grey = window_grey(volume_hu, width, level)
    return np.repeat(grey[..., None], 3, axis=-1)


volume_repeat = repeated_grey(volume_hu)

# sanity: the repeated version really does carry one channel's worth of information
print("three-window  channels identical:",
      np.array_equal(volume[..., 0], volume[..., 1]))
print("repeated      channels identical:",
      np.array_equal(volume_repeat[..., 0], volume_repeat[..., 1]))

mid = len(volume) // 2
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
axes[0].imshow(volume[mid]); axes[0].set_title("three windows", fontsize=11)
axes[1].imshow(volume_repeat[mid]); axes[1].set_title("one window, repeated", fontsize=11)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig("encoding_ab.png", dpi=150, bbox_inches="tight")

In [ ]:
# Same slices, same question, both encodings.
AB = {}
if model is not None and ceiling:
    idx = sample_indices(len(volume), ceiling)
    for name, vol in (("three windows", volume), ("repeated greyscale", volume_repeat)):
        free()
        try:
            inputs = processor.apply_chat_template(
                build_messages(vol[idx], QUESTION), add_generation_prompt=True,
                continue_final_message=False, return_tensors="pt",
                tokenize=True, return_dict=True,
            ).to(model.device)
            with torch.inference_mode():
                out = model.generate(**inputs, do_sample=False, max_new_tokens=300)
            AB[name] = processor.post_process_image_text_to_text(
                out, skip_special_tokens=True)[0]
        except Exception as e:
            AB[name] = f"[failed: {type(e).__name__}: {str(e)[:100]}]"

    for name, text in AB.items():
        print(f"{'=' * 70}\n{name.upper()}  ({ceiling} slices)\n{'=' * 70}")
        print(text.strip()[-1800:])
        print()
else:
    print("needs a working model and a non-zero ceiling")

Read the two answers side by side. The things worth noting for the write-up:

- do they identify the same findings, on the same slice numbers?
- is one more specific, or more hedged?
- does either invent something the other does not?

A single case cannot settle which encoding is better in general — but if the answers are
indistinguishable, that is worth knowing before you add windowing complexity to a
pipeline.

---

Project: https://github.com/rekalantar/MedGemma-CT-Reader

The windowing, sampling and prompt-building above are also an installable package in that
repo, with tests asserting they match MedGemma's own CT notebook byte for byte.